In [1]:
!nvidia-smi --query-gpu=memory.total --format=csv,noheader


16384 MiB


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" 
os.environ["KERAS_BACKEND"] = "tensorflow"
# tf.debugging.set_log_device_placement(True)

import tensorflow as tf
import keras
import tensorflow_datasets as tfds
import numpy as np
# import keras_hub


2026-05-08 04:49:52.302558: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-08 04:49:52.302618: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-08 04:49:52.304783: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Split Single Device into 2

In [3]:
physical_devices = tf.config.list_physical_devices("GPU")
tf.config.set_logical_device_configuration(
    physical_devices[0],
    [
        tf.config.LogicalDeviceConfiguration(memory_limit=15360 // 2),
        tf.config.LogicalDeviceConfiguration(memory_limit=15360 // 2),
    ],
)

logical_devices = tf.config.list_logical_devices("GPU")
logical_devices 

[LogicalDevice(name='/device:GPU:0', device_type='GPU'),
 LogicalDevice(name='/device:GPU:1', device_type='GPU')]

## Distributed Training Strategies in Tensorflow
- Sync vs Async
- Platform specific

Startegy types:
- MirroredStrategy
- TPUStrategy
- MultiWorkerMirroredStrategy
- ParameterServerStrategy
- CentralStorageStrategy

In [4]:
if tf.config.list_physical_devices('GPU'):
  strategy = tf.distribute.MirroredStrategy()
else:  # Use the Default Strategy
  strategy = tf.distribute.get_strategy()



INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


In [5]:

print("\nsetting variables on each device\n")
with strategy.scope():
    v = tf.Variable(1.)
    print(f"{v}")

print("\ngetting devices for variables\n")

for i, val in enumerate(v.values):
    print(i, val.numpy(), val.device)


setting variables on each device

MirroredVariable:{
  0: <tf.Variable 'Variable:0' shape=() dtype=float32, numpy=1.0>,
  1: <tf.Variable 'Variable/replica_1:0' shape=() dtype=float32, numpy=1.0>
}

getting devices for variables

0 1.0 /job:localhost/replica:0/task:0/device:GPU:0
1 1.0 /job:localhost/replica:0/task:0/device:GPU:1


## Implementation on Model Training

In [ ]:

(train_ds_CIFAR, test_ds_CIFAR), info = tfds.load(
    "cifar10",
    split=["train", "test"],
    as_supervised=True,
    with_info=True,
    data_dir="/app/datasets/tfds",
)

CIFAR_CLASS = info.features["label"].names


CIFAR-10 Class Names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [7]:
VAL_SIZE_CIFAR=10000
BATCH_SIZE_CIFAR=64
RANDOM_SEED_CIFAR=67


train_ds_CIFAR = train_ds_CIFAR.shuffle(10000, seed=RANDOM_SEED_CIFAR, reshuffle_each_iteration=False)

# Split
val_ds_CIFAR   = train_ds_CIFAR.take(VAL_SIZE_CIFAR)
train_ds_CIFAR = train_ds_CIFAR.skip(VAL_SIZE_CIFAR)

In [8]:
def preprocess_ds_CIFAR(x, y):
    x_new = tf.cast(x, tf.float32) / 255.0
    y_new = tf.cast(y, tf.int32)
    return x_new, y_new

train_ds_CIFAR = train_ds_CIFAR.map(preprocess_ds_CIFAR, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_CIFAR = train_ds_CIFAR.batch(BATCH_SIZE_CIFAR)
train_ds_CIFAR = train_ds_CIFAR.prefetch(tf.data.AUTOTUNE)

val_ds_CIFAR = val_ds_CIFAR.map(preprocess_ds_CIFAR, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_CIFAR = val_ds_CIFAR.batch(BATCH_SIZE_CIFAR)
val_ds_CIFAR = val_ds_CIFAR.prefetch(tf.data.AUTOTUNE)

test_ds_CIFAR = test_ds_CIFAR.map(preprocess_ds_CIFAR, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_CIFAR = test_ds_CIFAR.batch(BATCH_SIZE_CIFAR)
test_ds_CIFAR = test_ds_CIFAR.prefetch(tf.data.AUTOTUNE)

In [ ]:
print("Num replicas:", strategy.num_replicas_in_sync)
print("Logical devices:", tf.config.list_logical_devices("GPU"))

Num replicas: 2
Logical devices: [LogicalDevice(name='/device:GPU:0', device_type='GPU'), LogicalDevice(name='/device:GPU:1', device_type='GPU')]


In [10]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

with strategy.scope():
    model = keras.Sequential([
        keras.Input(shape=(32,32,3)),

        # Block 1
        keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPool2D(2),
        keras.layers.Dropout(0.2),
        # Block 2
        keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPool2D(2),
        keras.layers.Dropout(0.4),
        # Block 3
        keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPool2D(2),
        keras.layers.Dropout(0.5),
        # Classifier
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(10, activation="softmax")
    ], name="mlp_object_classifier")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False
    )
    early_stopping = EarlyStopping(monitor='val_loss', patience=8, verbose=1, mode='min')
    
    result = model.fit(
        train_ds_CIFAR,
        epochs=30, 
        validation_data=val_ds_CIFAR, 
        callbacks=[early_stopping],
    )

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


...

Epoch 1/30
INFO:tensorflow:Collective all_reduce tensors: 30 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.RING, num_packs = 1


INFO:tensorflow:Collective all_reduce tensors: 30 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.RING, num_packs = 1


INFO:tensorflow:Collective all_reduce tensors: 30 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.RING, num_packs = 1


INFO:tensorflow:Collective all_reduce tensors: 30 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.RING, num_packs = 1
2026-05-08 04:50:08.775545: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:1021] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmlp_object_classifier/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1778215812.844002 1344357 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 [==============================] - 37s 32ms/step - loss: 1.8044 - accuracy: 0.3922 - val_loss: 1.4121 - val_accuracy: 0.4997
Epoch 2/30
625/625 [==============================] - 19s 31ms/step - loss: 1.2059 - accuracy: 0.5717 - val_loss: 0.9783 - val_accuracy: 0.6582
Epoch 3/30
625/625 [==============================] - 19s 29ms/step - loss: 1.0260 - accuracy: 0.6409 - val_loss: 0.8977 - val_accuracy: 0.6863
Epoch 4/30
625/625 [==============================] - 19s 30ms/step - loss: 0.9219 - accuracy: 0.6781 - val_loss: 0.8078 - val_accuracy: 0.7128
Epoch 5/30
625/625 [==============================] - 19s 30ms/step - loss: 0.8446 - accuracy: 0.7082 - val_loss: 1.5511 - val_accuracy: 0.5180
Epoch 6/30
625/625 [==============================] - 19s 30ms/step - loss: 0.7853 - accuracy: 0.7290 - val_loss: 0.7701 - val_accuracy: 0.7347
Epoch 7/30
625/625 [==============================] - 19s 30ms/step - loss: 0.7344 - accuracy: 0.7469 - val_loss: 1.1725 - val_accuracy: 0.6189
Epo

In [11]:
for d in logical_devices:
    info = tf.config.experimental.get_memory_info(d.name)
    print(f"{d.name} current: {info['current'] / 1024**3} GB peak: {info['peak'] / 1024**3} GB")

/device:GPU:0 current: 0.006222248077392578 GB peak: 0.14075207710266113 GB
/device:GPU:1 current: 0.006201028823852539 GB peak: 0.1432180404663086 GB
